# 🎓 Project 03 — Student Management System (SQLite + OOP + SQL)

**Skills used:** `sqlite3`, SQL (CREATE / INSERT / SELECT / UPDATE / DELETE / JOIN / GROUP BY), parameterised queries (SQL-injection safety), context managers, transactions, classes, CSV export.

A database gives you what files can't: fast queries, relationships, integrity constraints and atomic transactions. SQLite needs no server — the whole database is one file (or `:memory:`).

---
## Step 1 — Design the schema
```
students(id PK, name, email UNIQUE, dept_id -> departments.id)
departments(id PK, name UNIQUE)
courses(id PK, title, credits)
marks(student_id -> students.id, course_id -> courses.id, score 0-100, PRIMARY KEY(student_id, course_id))
```

In [1]:
import sqlite3, csv, io

SCHEMA = """
PRAGMA foreign_keys = ON;
CREATE TABLE IF NOT EXISTS departments (
    id   INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL UNIQUE
);
CREATE TABLE IF NOT EXISTS students (
    id      INTEGER PRIMARY KEY AUTOINCREMENT,
    name    TEXT NOT NULL,
    email   TEXT NOT NULL UNIQUE,
    dept_id INTEGER REFERENCES departments(id) ON DELETE SET NULL
);
CREATE TABLE IF NOT EXISTS courses (
    id      INTEGER PRIMARY KEY AUTOINCREMENT,
    title   TEXT NOT NULL UNIQUE,
    credits INTEGER NOT NULL CHECK (credits > 0)
);
CREATE TABLE IF NOT EXISTS marks (
    student_id INTEGER NOT NULL REFERENCES students(id) ON DELETE CASCADE,
    course_id  INTEGER NOT NULL REFERENCES courses(id)  ON DELETE CASCADE,
    score      REAL    NOT NULL CHECK (score BETWEEN 0 AND 100),
    PRIMARY KEY (student_id, course_id)
);
"""
print(SCHEMA.count("CREATE TABLE"), "tables defined")

4 tables defined


## Step 2 — A repository class hiding the SQL

In [2]:
class SchoolDB:
    def __init__(self, path=":memory:"):
        self.con = sqlite3.connect(path)
        self.con.row_factory = sqlite3.Row            # rows behave like dicts
        self.con.executescript(SCHEMA)

    # `with self.con:` = a transaction: commit on success, rollback on error
    def add_department(self, name):
        with self.con:
            cur = self.con.execute("INSERT OR IGNORE INTO departments(name) VALUES (?)", (name,))
        return self.con.execute("SELECT id FROM departments WHERE name = ?", (name,)).fetchone()["id"]

    def add_student(self, name, email, dept):
        dept_id = self.add_department(dept)
        try:
            with self.con:
                cur = self.con.execute("INSERT INTO students(name, email, dept_id) VALUES (?, ?, ?)", (name, email, dept_id))
            return cur.lastrowid
        except sqlite3.IntegrityError as e:
            raise ValueError(f"Cannot add {name}: {e}") from e

    def add_course(self, title, credits):
        with self.con:
            cur = self.con.execute("INSERT OR IGNORE INTO courses(title, credits) VALUES (?, ?)", (title, credits))
        return self.con.execute("SELECT id FROM courses WHERE title = ?", (title,)).fetchone()["id"]

    def record_mark(self, student_id, course_id, score):
        with self.con:                                 # UPSERT: insert or update
            self.con.execute(
                "INSERT INTO marks(student_id, course_id, score) VALUES (?, ?, ?) "
                "ON CONFLICT(student_id, course_id) DO UPDATE SET score = excluded.score",
                (student_id, course_id, score))

    def delete_student(self, student_id):
        with self.con:
            return self.con.execute("DELETE FROM students WHERE id = ?", (student_id,)).rowcount

    def query(self, sql, params=()):
        return [dict(r) for r in self.con.execute(sql, params)]

db = SchoolDB()
print("database ready")

database ready


## Step 3 — Insert sample data

In [3]:
people = [("Asha Rao", "asha@uni.edu", "CSE"), ("Ravi Kumar", "ravi@uni.edu", "CSE"),
          ("Meena Iyer", "meena@uni.edu", "ECE"), ("Tom Joseph", "tom@uni.edu", "ECE"),
          ("Kiran Shah", "kiran@uni.edu", "MECH")]
ids = {name: db.add_student(name, email, dept) for name, email, dept in people}
courses = {"Python": db.add_course("Python", 4), "Maths": db.add_course("Maths", 3), "Physics": db.add_course("Physics", 3)}

scores = {"Asha Rao": (95, 88, 79), "Ravi Kumar": (72, 65, 58), "Meena Iyer": (85, 91, 84),
          "Tom Joseph": (48, 52, 39), "Kiran Shah": (77, 70, 82)}
for name, marks in scores.items():
    for course_id, score in zip(courses.values(), marks):
        db.record_mark(ids[name], course_id, score)

print(db.query("SELECT COUNT(*) AS students FROM students"), db.query("SELECT COUNT(*) AS marks FROM marks"))

[{'students': 5}] [{'marks': 15}]


## Step 4 — Queries: SELECT, JOIN, GROUP BY, HAVING, window-like ranking

In [4]:
def show(title, rows):
    print(f"\n▶ {title}")
    if not rows:
        print("  (no rows)"); return
    cols = list(rows[0])
    print("  " + " | ".join(f"{c:<12}" for c in cols))
    for r in rows:
        print("  " + " | ".join(f"{str(r[c]):<12}" for c in cols))

show("Student transcript (JOIN across 3 tables)", db.query("""
    SELECT s.name, c.title AS course, m.score
    FROM marks m
    JOIN students s ON s.id = m.student_id
    JOIN courses  c ON c.id = m.course_id
    WHERE s.name = ?
    ORDER BY m.score DESC""", ("Asha Rao",)))

show("Average score per student (GROUP BY) with ranking", db.query("""
    SELECT s.name, d.name AS dept, ROUND(AVG(m.score), 1) AS average
    FROM students s
    JOIN departments d ON d.id = s.dept_id
    JOIN marks m ON m.student_id = s.id
    GROUP BY s.id
    ORDER BY average DESC"""))

show("Department statistics", db.query("""
    SELECT d.name AS dept, COUNT(DISTINCT s.id) AS students, ROUND(AVG(m.score), 1) AS avg_score, MAX(m.score) AS best
    FROM departments d
    JOIN students s ON s.dept_id = d.id
    JOIN marks m ON m.student_id = s.id
    GROUP BY d.id
    ORDER BY avg_score DESC"""))

show("Students failing at least one course (< 40)", db.query("""
    SELECT DISTINCT s.name, c.title AS course, m.score
    FROM marks m JOIN students s ON s.id = m.student_id JOIN courses c ON c.id = m.course_id
    WHERE m.score < 40"""))

show("Course toppers (correlated subquery)", db.query("""
    SELECT c.title AS course, s.name, m.score
    FROM marks m
    JOIN students s ON s.id = m.student_id
    JOIN courses c ON c.id = m.course_id
    WHERE m.score = (SELECT MAX(score) FROM marks WHERE course_id = m.course_id)
    ORDER BY c.title"""))

show("Weighted GPA-like score (credits as weights)", db.query("""
    SELECT s.name, ROUND(SUM(m.score * c.credits) * 1.0 / SUM(c.credits), 2) AS weighted
    FROM marks m JOIN students s ON s.id = m.student_id JOIN courses c ON c.id = m.course_id
    GROUP BY s.id HAVING weighted >= 70 ORDER BY weighted DESC"""))


▶ Student transcript (JOIN across 3 tables)
  name         | course       | score       
  Asha Rao     | Python       | 95.0        
  Asha Rao     | Maths        | 88.0        
  Asha Rao     | Physics      | 79.0        

▶ Average score per student (GROUP BY) with ranking
  name         | dept         | average     
  Asha Rao     | CSE          | 87.3        
  Meena Iyer   | ECE          | 86.7        
  Kiran Shah   | MECH         | 76.3        
  Ravi Kumar   | CSE          | 65.0        
  Tom Joseph   | ECE          | 46.3        

▶ Department statistics
  dept         | students     | avg_score    | best        
  MECH         | 1            | 76.3         | 82.0        
  CSE          | 2            | 76.2         | 95.0        
  ECE          | 2            | 66.5         | 91.0        

▶ Students failing at least one course (< 40)
  name         | course       | score       
  Tom Joseph   | Physics      | 39.0        

▶ Course toppers (correlated subquery)
  course  

## Step 5 — Constraints protect your data

In [5]:
tests = [
    ("duplicate email", lambda: db.add_student("Clone", "asha@uni.edu", "CSE")),
    ("score out of range", lambda: db.record_mark(ids["Asha Rao"], courses["Python"], 150)),
    ("non-existent student", lambda: db.record_mark(9999, courses["Python"], 50)),
    ("credits <= 0", lambda: db.con.execute("INSERT INTO courses(title, credits) VALUES ('Bad', 0)")),
]
for label, action in tests:
    try:
        action()
        print(f"{label}: accepted?!")
    except (ValueError, sqlite3.IntegrityError) as e:
        print(f"{label}: rejected ✔  ({type(e).__name__})")

# SQL injection: parameterised queries treat input as DATA, never as SQL
evil = "x' OR '1'='1"
print("safe lookup with evil input ->", db.query("SELECT * FROM students WHERE name = ?", (evil,)))
# (A string-formatted query f"... WHERE name = '{evil}'" would return EVERY student!)

# Cascade delete: removing a student removes their marks
before = db.query("SELECT COUNT(*) AS n FROM marks")[0]["n"]
db.delete_student(ids["Tom Joseph"])
after = db.query("SELECT COUNT(*) AS n FROM marks")[0]["n"]
print("marks rows before/after deleting a student:", before, "->", after)

duplicate email: rejected ✔  (ValueError)
score out of range: rejected ✔  (IntegrityError)
non-existent student: rejected ✔  (IntegrityError)
credits <= 0: rejected ✔  (IntegrityError)
safe lookup with evil input -> []
marks rows before/after deleting a student: 15 -> 12


## Step 6 — Transactions: all-or-nothing

In [6]:
def transfer_marks_batch(entries):
    """Apply several marks atomically: if ANY fails, none are saved."""
    try:
        with db.con:
            for student_id, course_id, score in entries:
                db.con.execute("INSERT OR REPLACE INTO marks VALUES (?, ?, ?)", (student_id, course_id, score))
        return "committed"
    except sqlite3.IntegrityError:
        return "rolled back"

before = db.query("SELECT ROUND(AVG(score), 2) AS a FROM marks")[0]["a"]
print(transfer_marks_batch([(ids["Asha Rao"], courses["Python"], 100), (ids["Ravi Kumar"], courses["Maths"], 999)]))
after = db.query("SELECT ROUND(AVG(score), 2) AS a FROM marks")[0]["a"]
print("average unchanged after failed batch:", before == after)

rolled back
average unchanged after failed batch: True


## Step 7 — Reports: export to CSV and use pandas

In [7]:
rows = db.query("SELECT s.name, ROUND(AVG(m.score),1) AS average FROM students s JOIN marks m ON m.student_id=s.id GROUP BY s.id ORDER BY average DESC")
buf = io.StringIO()
w = csv.DictWriter(buf, fieldnames=["name", "average"])
w.writeheader(); w.writerows(rows)
print(buf.getvalue())

import pandas as pd
df = pd.read_sql_query("SELECT s.name, c.title, m.score FROM marks m JOIN students s ON s.id=m.student_id JOIN courses c ON c.id=m.course_id", db.con)
print(df.pivot(index="name", columns="title", values="score"))

name,average
Asha Rao,87.3
Meena Iyer,86.7
Kiran Shah,76.3
Ravi Kumar,65.0

title       Maths  Physics  Python
name                              
Asha Rao     88.0     79.0    95.0
Kiran Shah   70.0     82.0    77.0
Meena Iyer   91.0     84.0    85.0
Ravi Kumar   65.0     58.0    72.0


---
## 🚀 Extensions
1. Add a `teachers` table and a many-to-many `teaches` relation.
2. Build a menu-driven CLI (`while True:` + `input`) around `SchoolDB`.
3. Add grade letters with `CASE WHEN score >= 90 THEN 'A' ...` inside SQL.
4. Add an index (`CREATE INDEX idx_marks_course ON marks(course_id)`) and compare `EXPLAIN QUERY PLAN` before/after.
5. Wrap the database with a **Flask/FastAPI** web API.

**What you practised:** relational design, SQL, safe parameterisation, transactions, constraints, and connecting SQL ↔ Python ↔ pandas.